## Consignas

- Explicar qué es un socket y los diferentes tipos de sockets.
- Cuáles son las estructuras necesarias para operar con sockets en el modelo C-S y cómo se hace para ingresar los datos requeridos.
-- Explicar con un ejemplo.
- Explicar modo bloqueante y no bloqueante en sockets y cuáles son los “sockets calls” a los cuales se pueden aplicar estos modos.
-- Explicar las funciones necesarias.
- Describir el modelo C-S aplicado a un servidor con Concurrencia Real.
- Escribir un ejemplo en lenguaje C distinto del ya presentado.
- Cómo se cierra una conexión C-S ?. Métodos que eviten la pérdida de informacion.
- Probar el código del chat entre cliente y servidor. Cambiar tipo de socket y volver a probar.
- Presentar las respuestas en archivos Jupyter Notebook o PDF.

### Respustas
- Un socket representa el punto final de una conexion entre dos dispositivos, este es la combinacion numerica entre la ip y un puerto establecido. A traves de esta interfaz, las aplicaciones pueden intercambiar datos.
  Existen diferentes tipos de sockets dependiendo el protocolo de transporte que se utilice en la conexion. Esta el Socket tipo Stream,
  utilizado para conexiones TCP y los tipos DAtagram asociados a las conexiones de tipo UDP. A su vez existe un tipo llamado RAW (crudo),
  el cual permite una conexion a mas bajo nivel donde no utiliza ninguno de los protocolos mencionados y permite leer y escribir al nivel de la capa de red
- Se utiliza la estructura sockaddr_in, la cual representa las direcciones en ipv4 utilizadas en internet aunque al utilizar una funcion del sistema (bind, sendto, connect, etc.) se castea con la estructura sockaddr dado que el sistema necesita una interfaz generica e independiente del protocolo. Esta contiene el tipo de familia de direccion (en caso de _in usa siempre ipv4, es decir AF_INET), El puerto, un padding para rellenar y una struct in_addr que contiene la direccion ip. Al cargar el puierto es necesario utilizar la funcion htons (host to network short) convirtiendo los bits a formato de red (big endian)
ejemplo:
```c
struct sockaddr_in server_addr;

server_addr.sin_family = AF_INET;
server_addr.sin_port = htons(8080);
inet_pton(AF_INET, "127.0.0.1", &server_addr.sin_addr);
memset(server_addr.sin_zero, 0, 8);
```
- El modo bloqueante y el modo no bloqueante son los modos de operacion del socket, que determinan como se comoportan las llamadas del sistema cuando los datos no estan inmediatamente disponibles.\
  En el modo bloqueante, cuando una llamada la sistema no puede completarse, el proceso queda suspendido hasta que la operacion se pueda realizar, por ejemplo al utilizar un `recv()` si no hay informacion disponible en el socket, el proceso se detendria hasta que lleguen los datos.\
  En el modo no bloqueante, las llamadas al sistema retornan inmediatamente, incluso si no cumplen con la operacion esperada. En este caso, el sistema indica mediante un error que no pudo realizarce la accion, lo que permite que el proceso continue ejecutandose sin quedar detenido, pero requiere la implementacion de mecanismos adicionales para el control de los datos.\
  Estos modos pueden aplicarse a la mayoria de operaciones sobre sockets, entre ellas tenemos:\
  `accept()`: en modo bloqueante espera hasta que llegue una conexión entrante; en modo no bloqueante retorna inmediatamente si no hay
  clientes.\
  `recv()` / `recvfrom()`: en modo bloqueante esperan datos; en modo no bloqueante retornan si no hay datos disponibles.\
  `read()`: equivalente a recv en muchos casos, tambien puede bloquear.\
  `send()` / `sendto()` /`write()`: pueden bloquear si los buffers de envio están llenos; en modo no bloqueante retornan si no pueden
  enviar en ese momento.\
  `connect()`: en modo bloqueante espera hasta establecer la conexion; en modo no bloqueante retorna inmediatamente mientras la conexion se
  establece en segundo plano.\
- Un servidor con Concurerencia real es aquel que puede manejar de manera simultanea a multiples clientes simultaneamente. La idea es que cada cliente es manejado por una unidad de ejecucion independentiente, de modo que lo que suceda con un cliente no afecta a otros. \
  El servidor crea primero un socket, lo asocia a una direccion y queda a la espera de un cliente. La concurrencia aparee a la hora de llegar varios clientes; en lugar de atenderlos uno por uno de manera secuacial, el servidor crea un proceso hijo mediante la funcion `fork()` donde cada proceso hijo se encarga de interactuar con un cliente en particular mientras el proceso padre se queda aceptando nuevas conexiones. La funcion `fork()` copia las variables, datos y los descriptores de archivo de los sockets, todo en un espacio de memoria diferente, permitiendo la concurrencia real ejecutandose simultaneamente en espacios de memoria diferentes. La funcion retorna un Id del proceso,  `(-1)` para errores,  `0` indica que es el hijo, `>0` indica que nos encontramos en el padre. De esta forma podemos organizar las tareas que realiza el padre y los procesos hijos.

```c
/*Este codigo muestra un ejemplo del uso del fork diferente al presentado,
  En este ejemplo se plantea un modelo prefork donde cada hijo se encarga completamente
  de la comunicacion con los clientes mientras el padres solamente gestiona los procesos.
*/
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <netinet/in.h>
#include <arpa/inet.h>
#include <signal.h>

#define MAX 1024
#define PORT 8080

void worker(int server_fd) {
    char buffer[MAX];
    char name[50] = "Cliente";
    char client_ip[INET_ADDRSTRLEN];
    char response[MAX + 100];
    int client_fd;
     struct sockaddr_in client_addr;
    int pid = getpid();
     
 while (1){
        socklen_t len = sizeof(client_addr);
        client_fd = accept(server_fd, (struct sockaddr*)&client_addr, &len);
            if (client_fd < 0) {
                perror("Fallo en accept");
                continue;
            }


        inet_ntop(AF_INET, &(client_addr.sin_addr), client_ip, INET_ADDRSTRLEN);
        int client_port = ntohs(client_addr.sin_port);

        printf("[PID %d] Conexión desde %s:%d\n", pid, client_ip, client_port);

        // Recibir nombre del cliente
        recv(client_fd, name, sizeof(name), 0);
        printf("[PID %d] Cliente: %s\n", pid, name);

        while (1) {
            memset(buffer, 0, MAX);
            int bytes = recv(client_fd, buffer, MAX, 0);
            if (bytes <= 0) {
                printf("[PID %d] %s se desconectó.\n", pid, name);
                break;
            }

            // Verificar si cliente quiere finalizar
            if (strncmp(buffer, "FIN", 3) == 0) {
                printf("[PID %d] %s finalizó la conexión.\n", pid, name);
                break;
            }

            printf("[%s | PID %d]: %s", name, pid, buffer);

            // Preparar respuesta automática
            snprintf(response, sizeof(response),
                     "Este es el mensaje recibido por el servidor: %s", buffer);
            send(client_fd, response, strlen(response), 0);
        }

        close(client_fd);
        }
}

int main() {
    int server_fd;
    struct sockaddr_in server_addr;

    signal(SIGCHLD, SIG_IGN); // Evita procesos zombies

    server_fd = socket(AF_INET, SOCK_STREAM, 0);
    if (server_fd == -1) {
        perror("Error al crear socket");
        exit(1);
    }

    server_addr.sin_family = AF_INET;
    server_addr.sin_addr.s_addr = INADDR_ANY;
    server_addr.sin_port = htons(PORT);
    memset(&(server_addr.sin_zero), 0, 8);

    if (bind(server_fd, (struct sockaddr*)&server_addr, sizeof(server_addr)) != 0) {
        perror("Error en bind");
        exit(1);
    }

    if (listen(server_fd, 5) != 0) {
        perror("Error en listen");
        exit(1);
    }

    printf("Servidor concurrente activo en puerto %d...\n", PORT);

    
   
    for(int i =0; i<5;i++) {
     int pid = fork();
        if (pid ==0){
            worker(server_fd); 
            exit(0);
        }
        else if(pid >0){
             printf("Proceso hijo creado. PID[%d]",pid);
        } else {
            perror("Error en fork");
    }
}
    while(1){
    pause();
    }
    return 0;
}
```

- Existen dos formas de cerrar la conexion en el modelo C-S. Una es utilizando la funcion `shutdown()`, la cual permite cerrar la comunicacion en una de las dos vias, permitiendo todavia recibir informacion pero indicando que no va a transmitir mas. \
```c
int shutdown(int sockfd, int how);
```
 donde el parametro how representa `how = 0` no se permite recibir mas datos, `how = 1` no se permiten enviar mas datos. Esta ultima opcion es la mas segura y la que impide perdida de informacion, ya que el socket queda operativo para poder recibir informacion que del otro extremo aun se esta emitiendo. con `how = 2` cumple una funcion similar a `close()` . \
 La otra forma utiliza la funcion `close()` el cual cierra la conexion indicando que no envia mas y que no recibira mas tampoco. \
La forma correcta de finalizar una conexion es definir un protocolo de cierre para que ninguna de las partes envie o espere informacion en un momento que no corresponda, como la que implementamos de finalizar la conexion con un "FIN". utilizando `shutdown()` con `how = 1`, para garantizar que la informacion que el otro extremo aun esta enviando se recibira de forma correcta. Recalcar tambien que `shutdown()` no libera realmente el descriptor del socket, sino que solo cambia sus condiciones de uso, para liberar el recurso correctamente hay que complementar usnado la funcion `close()`.
- Se hizo el cambio en los sockets, reemplazando `socket(AF_INET, SOCK_STREAM, 0);` por `socket(AF_INET, SOCK_DGRAM, 0);`. Esto altera el protocolo de comunicacion entre el C-S sin cambiar la arquitectura del codigo. La comunicacion sigue funcionando demostrando que el protocolo de transporte es independiente de la comunicacion. Logicamente una comunicacion controlada por TCP proporciona que el chat sea mas fiable.